# Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE
import joblib
import json
import os

print("✅ All libraries imported!")

✅ All libraries imported!


# Sample data

In [2]:
# ============================================
# CREATE SAMPLE DATA FOR TESTING (FIXED)
# ============================================

import pandas as pd
import numpy as np

print("="*70)
print("CREATING SAMPLE DATA (For Testing)")
print("="*70)
print()
print("⚠️  Using synthetic data instead of real datasets")
print("   This allows testing without downloading large files")
print()

# Create realistic sample UNSW-NB15 data
n_samples = 50000  # 50k samples per dataset

np.random.seed(42)

# UNSW-NB15 Dataset
print("Creating UNSW-NB15 sample...")
unsw_df = pd.DataFrame({
    'dur': np.random.exponential(0.5, n_samples),
    'proto': np.random.choice(['tcp', 'udp', 'icmp', 'arp'], n_samples, p=[0.7, 0.2, 0.08, 0.02]),
    'service': np.random.choice(['http', 'ftp', 'ssh', 'smtp', 'dns', '-'], n_samples, p=[0.4, 0.1, 0.1, 0.05, 0.15, 0.2]),
    'state': np.random.choice(['FIN', 'INT', 'CON', 'REQ', 'RST'], n_samples, p=[0.3, 0.2, 0.25, 0.15, 0.1]),
    'spkts': np.random.poisson(10, n_samples),
    'dpkts': np.random.poisson(8, n_samples),
    'sbytes': np.random.exponential(1000, n_samples).astype(int),
    'dbytes': np.random.exponential(800, n_samples).astype(int),
    'rate': np.random.exponential(50, n_samples),
    'sttl': np.random.randint(0, 256, n_samples),
    'dttl': np.random.randint(0, 256, n_samples),
    'sload': np.random.exponential(100, n_samples),
    'dload': np.random.exponential(80, n_samples),
    'sloss': np.random.poisson(0.5, n_samples),
    'dloss': np.random.poisson(0.5, n_samples),
    'sinpkt': np.random.exponential(20, n_samples),
    'dinpkt': np.random.exponential(20, n_samples),
    'sjit': np.random.exponential(10, n_samples),
    'djit': np.random.exponential(10, n_samples),
    'swin': np.random.randint(0, 65536, n_samples),
    'stcpb': np.random.randint(0, 2147483647, n_samples, dtype=np.int64),  # FIXED
    'dtcpb': np.random.randint(0, 2147483647, n_samples, dtype=np.int64),  # FIXED
    'dwin': np.random.randint(0, 65536, n_samples),
    'tcprtt': np.random.exponential(50, n_samples),
    'synack': np.random.exponential(30, n_samples),
    'ackdat': np.random.exponential(40, n_samples),
    'smean': np.random.exponential(500, n_samples),
    'dmean': np.random.exponential(400, n_samples),
    'trans_depth': np.random.poisson(2, n_samples),
    'response_body_len': np.random.exponential(2000, n_samples).astype(int),
    'ct_srv_src': np.random.poisson(3, n_samples),
    'ct_state_ttl': np.random.poisson(2, n_samples),
    'ct_dst_ltm': np.random.poisson(4, n_samples),
    'ct_src_dport_ltm': np.random.poisson(3, n_samples),
    'ct_dst_sport_ltm': np.random.poisson(3, n_samples),
    'ct_dst_src_ltm': np.random.poisson(5, n_samples),
    'is_ftp_login': np.random.choice([0, 1], n_samples, p=[0.95, 0.05]),
    'ct_ftp_cmd': np.random.poisson(0.5, n_samples),
    'ct_flw_http_mthd': np.random.poisson(1, n_samples),
    'ct_src_ltm': np.random.poisson(4, n_samples),
    'ct_srv_dst': np.random.poisson(3, n_samples),
    'is_sm_ips_ports': np.random.choice([0, 1], n_samples, p=[0.9, 0.1]),
    'label': np.random.choice([0, 1], n_samples, p=[0.8, 0.2])
})

print(f"✅ UNSW-NB15 created: {unsw_df.shape}")
print(f"   Normal: {(unsw_df['label']==0).sum()}")
print(f"   Attack: {(unsw_df['label']==1).sum()}")
print()

# CICIDS2017 Dataset
print("Creating CICIDS2017 sample...")
cic_df = pd.DataFrame({
    'Flow Duration': np.random.exponential(10000, n_samples).astype(int),
    'Total Fwd Packets': np.random.poisson(10, n_samples),
    'Total Backward Packets': np.random.poisson(8, n_samples),
    'Total Length of Fwd Packets': np.random.exponential(5000, n_samples).astype(int),
    'Total Length of Bwd Packets': np.random.exponential(4000, n_samples).astype(int),
    'Fwd Packet Length Max': np.random.exponential(1500, n_samples),
    'Fwd Packet Length Min': np.random.exponential(100, n_samples),
    'Fwd Packet Length Mean': np.random.exponential(500, n_samples),
    'Fwd Packet Length Std': np.random.exponential(200, n_samples),
    'Bwd Packet Length Max': np.random.exponential(1500, n_samples),
    'Bwd Packet Length Min': np.random.exponential(100, n_samples),
    'Bwd Packet Length Mean': np.random.exponential(500, n_samples),
    'Bwd Packet Length Std': np.random.exponential(200, n_samples),
    'Flow Bytes/s': np.random.exponential(10000, n_samples),
    'Flow Packets/s': np.random.exponential(100, n_samples),
    'Flow IAT Mean': np.random.exponential(1000, n_samples),
    'Flow IAT Std': np.random.exponential(500, n_samples),
    'Flow IAT Max': np.random.exponential(5000, n_samples),
    'Flow IAT Min': np.random.exponential(10, n_samples),
    'Fwd IAT Total': np.random.exponential(5000, n_samples),
    'Fwd IAT Mean': np.random.exponential(500, n_samples),
    'Fwd IAT Std': np.random.exponential(300, n_samples),
    'Fwd IAT Max': np.random.exponential(2000, n_samples),
    'Fwd IAT Min': np.random.exponential(10, n_samples),
    'Bwd IAT Total': np.random.exponential(5000, n_samples),
    'Bwd IAT Mean': np.random.exponential(500, n_samples),
    'Bwd IAT Std': np.random.exponential(300, n_samples),
    'Bwd IAT Max': np.random.exponential(2000, n_samples),
    'Bwd IAT Min': np.random.exponential(10, n_samples),
    'Fwd PSH Flags': np.random.poisson(0.5, n_samples),
    'Bwd PSH Flags': np.random.poisson(0.5, n_samples),
    'Fwd URG Flags': np.random.poisson(0.1, n_samples),
    'Bwd URG Flags': np.random.poisson(0.1, n_samples),
    'Fwd Header Length': np.random.poisson(40, n_samples),
    'Bwd Header Length': np.random.poisson(40, n_samples),
    'Fwd Packets/s': np.random.exponential(50, n_samples),
    'Bwd Packets/s': np.random.exponential(40, n_samples),
    'Min Packet Length': np.random.exponential(60, n_samples),
    'Max Packet Length': np.random.exponential(1500, n_samples),
    'Packet Length Mean': np.random.exponential(500, n_samples),
    'Packet Length Std': np.random.exponential(300, n_samples),
    'Packet Length Variance': np.random.exponential(100000, n_samples),
    'FIN Flag Count': np.random.poisson(0.5, n_samples),
    'SYN Flag Count': np.random.poisson(0.5, n_samples),
    'RST Flag Count': np.random.poisson(0.2, n_samples),
    'PSH Flag Count': np.random.poisson(1, n_samples),
    'ACK Flag Count': np.random.poisson(5, n_samples),
    'URG Flag Count': np.random.poisson(0.1, n_samples),
    'CWE Flag Count': np.random.poisson(0.1, n_samples),
    'ECE Flag Count': np.random.poisson(0.1, n_samples),
    'Down/Up Ratio': np.random.exponential(1, n_samples),
    'Average Packet Size': np.random.exponential(500, n_samples),
    'Avg Fwd Segment Size': np.random.exponential(500, n_samples),
    'Avg Bwd Segment Size': np.random.exponential(400, n_samples),
    'Fwd Avg Bytes/Bulk': np.random.exponential(1000, n_samples),
    'Fwd Avg Packets/Bulk': np.random.poisson(3, n_samples),
    'Fwd Avg Bulk Rate': np.random.exponential(100, n_samples),
    'Bwd Avg Bytes/Bulk': np.random.exponential(1000, n_samples),
    'Bwd Avg Packets/Bulk': np.random.poisson(3, n_samples),
    'Bwd Avg Bulk Rate': np.random.exponential(100, n_samples),
    'Subflow Fwd Packets': np.random.poisson(10, n_samples),
    'Subflow Fwd Bytes': np.random.exponential(5000, n_samples).astype(int),
    'Subflow Bwd Packets': np.random.poisson(8, n_samples),
    'Subflow Bwd Bytes': np.random.exponential(4000, n_samples).astype(int),
    'Init_Win_bytes_forward': np.random.randint(0, 65536, n_samples),
    'Init_Win_bytes_backward': np.random.randint(0, 65536, n_samples),
    'act_data_pkt_fwd': np.random.poisson(5, n_samples),
    'min_seg_size_forward': np.random.poisson(20, n_samples),
    'Active Mean': np.random.exponential(1000, n_samples),
    'Active Std': np.random.exponential(500, n_samples),
    'Active Max': np.random.exponential(5000, n_samples),
    'Active Min': np.random.exponential(100, n_samples),
    'Idle Mean': np.random.exponential(10000, n_samples),
    'Idle Std': np.random.exponential(5000, n_samples),
    'Idle Max': np.random.exponential(50000, n_samples),
    'Idle Min': np.random.exponential(1000, n_samples),
    'binary_label': np.random.choice([0, 1], n_samples, p=[0.85, 0.15])
})

print(f"✅ CICIDS2017 created: {cic_df.shape}")
print(f"   Normal: {(cic_df['binary_label']==0).sum()}")
print(f"   Attack: {(cic_df['binary_label']==1).sum()}")
print()

print("="*70)
print("SAMPLE DATA READY!")
print("="*70)
print()
print("📊 You can now continue with preprocessing and training")
print("⚠️  Note: Models trained on sample data won't be production-ready")
print("   For real deployment, use actual UNSW-NB15 and CICIDS2017 datasets")
print()

CREATING SAMPLE DATA (For Testing)

⚠️  Using synthetic data instead of real datasets
   This allows testing without downloading large files

Creating UNSW-NB15 sample...
✅ UNSW-NB15 created: (50000, 43)
   Normal: 39889
   Attack: 10111

Creating CICIDS2017 sample...
✅ CICIDS2017 created: (50000, 77)
   Normal: 42458
   Attack: 7542

SAMPLE DATA READY!

📊 You can now continue with preprocessing and training
⚠️  Note: Models trained on sample data won't be production-ready
   For real deployment, use actual UNSW-NB15 and CICIDS2017 datasets



# Preprocess UNSW data

In [3]:
print("Preprocessing UNSW-NB15...")

# Separate features and labels
X_unsw = unsw_df.drop('label', axis=1)
y_unsw = unsw_df['label']

# Encode categorical columns
categorical_cols = ['proto', 'service', 'state']
label_encoders = {}

for col in categorical_cols:
    if col in X_unsw.columns:
        le = LabelEncoder()
        X_unsw[col] = le.fit_transform(X_unsw[col].astype(str))
        label_encoders[col] = le

# Split data
X_train_unsw, X_test_unsw, y_train_unsw, y_test_unsw = train_test_split(
    X_unsw, y_unsw, test_size=0.2, random_state=42, stratify=y_unsw
)

# Scale features
scaler_unsw = StandardScaler()
X_train_unsw_scaled = scaler_unsw.fit_transform(X_train_unsw)
X_test_unsw_scaled = scaler_unsw.transform(X_test_unsw)

# Balance with SMOTE
smote = SMOTE(random_state=42)
X_train_unsw_balanced, y_train_unsw_balanced = smote.fit_resample(
    X_train_unsw_scaled, y_train_unsw
)

print(f"✅ UNSW preprocessed!")
print(f"   Training samples: {X_train_unsw_balanced.shape[0]}")
print(f"   Test samples: {X_test_unsw_scaled.shape[0]}")

Preprocessing UNSW-NB15...
✅ UNSW preprocessed!
   Training samples: 63822
   Test samples: 10000


# Train UNSW models

In [4]:
print("Training UNSW models...")

# Random Forest
print("\n1. Random Forest...")
rf_unsw = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)
rf_unsw.fit(X_train_unsw_balanced, y_train_unsw_balanced)
y_pred_rf = rf_unsw.predict(X_test_unsw_scaled)
acc_rf = accuracy_score(y_test_unsw, y_pred_rf)
print(f"   Accuracy: {acc_rf*100:.2f}%")

# Gradient Boosting
print("\n2. Gradient Boosting...")
gb_unsw = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=5,
    random_state=42
)
gb_unsw.fit(X_train_unsw_balanced, y_train_unsw_balanced)
y_pred_gb = gb_unsw.predict(X_test_unsw_scaled)
acc_gb = accuracy_score(y_test_unsw, y_pred_gb)
print(f"   Accuracy: {acc_gb*100:.2f}%")

print("\n✅ UNSW models trained!")

Training UNSW models...

1. Random Forest...
   Accuracy: 79.78%

2. Gradient Boosting...
   Accuracy: 79.69%

✅ UNSW models trained!


# Preprocess CICIDS data

In [5]:
print("Preprocessing CICIDS2017...")

# Separate features and labels
X_cic = cic_df.drop('binary_label', axis=1)
y_cic = cic_df['binary_label']

# Split data
X_train_cic, X_test_cic, y_train_cic, y_test_cic = train_test_split(
    X_cic, y_cic, test_size=0.2, random_state=42, stratify=y_cic
)

# Scale features
scaler_cic = StandardScaler()
X_train_cic_scaled = scaler_cic.fit_transform(X_train_cic)
X_test_cic_scaled = scaler_cic.transform(X_test_cic)

# Balance with SMOTE
X_train_cic_balanced, y_train_cic_balanced = smote.fit_resample(
    X_train_cic_scaled, y_train_cic
)

print(f"✅ CICIDS preprocessed!")
print(f"   Training samples: {X_train_cic_balanced.shape[0]}")
print(f"   Test samples: {X_test_cic_scaled.shape[0]}")

Preprocessing CICIDS2017...
✅ CICIDS preprocessed!
   Training samples: 67932
   Test samples: 10000


# Train CICIDS models

In [6]:
print("Training CICIDS models...")

# Random Forest
print("\n1. Random Forest...")
rf_cic = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)
rf_cic.fit(X_train_cic_balanced, y_train_cic_balanced)
y_pred_rf_cic = rf_cic.predict(X_test_cic_scaled)
acc_rf_cic = accuracy_score(y_test_cic, y_pred_rf_cic)
print(f"   Accuracy: {acc_rf_cic*100:.2f}%")

# Gradient Boosting
print("\n2. Gradient Boosting...")
gb_cic = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    random_state=42
)
gb_cic.fit(X_train_cic_balanced, y_train_cic_balanced)
y_pred_gb_cic = gb_cic.predict(X_test_cic_scaled)
acc_gb_cic = accuracy_score(y_test_cic, y_pred_gb_cic)
print(f"   Accuracy: {acc_gb_cic*100:.2f}%")

print("\n✅ CICIDS models trained!")

Training CICIDS models...

1. Random Forest...
   Accuracy: 84.09%

2. Gradient Boosting...
   Accuracy: 84.92%

✅ CICIDS models trained!


# Save models

In [7]:
print("="*70)
print("SAVING MODELS")
print("="*70)

os.makedirs('models', exist_ok=True)

# Save models
joblib.dump(rf_unsw, 'models/random_forest_unsw.pkl')
joblib.dump(gb_unsw, 'models/gradient_boosting_unsw.pkl')
joblib.dump(rf_cic, 'models/random_forest_cicids.pkl')
joblib.dump(gb_cic, 'models/gradient_boosting_cicids.pkl')

# Save scalers
joblib.dump(scaler_unsw, 'models/scaler_unsw.pkl')
joblib.dump(scaler_cic, 'models/scaler_cic.pkl')

# Save feature names
with open('models/feature_names_unsw.json', 'w') as f:
    json.dump(list(X_unsw.columns), f)
    
with open('models/feature_names_cic.json', 'w') as f:
    json.dump(list(X_cic.columns), f)

print("✅ All models saved!")
print(f"   Location: {os.path.abspath('models/')}")
print()
print("🎉 TRAINING COMPLETE!")
print(f"   UNSW RF: {acc_rf*100:.2f}%")
print(f"   UNSW GB: {acc_gb*100:.2f}%")
print(f"   CIC RF: {acc_rf_cic*100:.2f}%")
print(f"   CIC GB: {acc_gb_cic*100:.2f}%")

SAVING MODELS
✅ All models saved!
   Location: C:\Users\nnamd\models

🎉 TRAINING COMPLETE!
   UNSW RF: 79.78%
   UNSW GB: 79.69%
   CIC RF: 84.09%
   CIC GB: 84.92%
